# Práctica guiada · Sesión 8: el flujo reproducible

Este notebook es la versión **reproducible** del análisis de la Sesión 7. Si lo reinicias y lo corres de la primera celda a la última, siempre da 3,200. Esa es la prueba: mismo insumo más mismos pasos igual a mismo resultado, para cualquiera.

Dos cosas lo hacen reproducible: cada arreglo al dato es una **línea de código**, no un clic que se hizo una vez; y cada bloque trae una **explicación** de qué hace y por qué. Eso es lo que en el pizarrón llamamos documentado.


## 1. El insumo, fijo y descrito

El mismo padrón de la Sesión 7: 18 registros de un programa de apoyo (ficticio), tal como llegan, todo como texto. No se toca a mano; todo lo que le pase, le pasa aquí.


In [1]:
import pandas as pd

# (folio, entidad, monto_apoyo tal como se capturo, estatus)
registro = [
    ("F-001","Norte","2000","Activo"),   ("F-002","Centro","5,000","activo"),
    ("F-003","Sur","2000","ACTIVO"),      ("F-004","Norte","5,000","Activo"),
    ("F-005","Centro","2000","Baja"),     ("F-006","Sur","2000","Activo "),
    ("F-007","Norte","5,000","Activo"),   ("F-008","Centro","2000","baja"),
    ("F-009","Sur","2000","Activo"),      ("F-010","Norte","5,000","Activo"),
    ("F-011","Centro","2000","Activo"),   ("F-012","Sur","5,000","Activo"),
    ("F-013","Norte","2000","Activo"),    ("F-014","Centro","5,000","Activo"),
    ("F-015","Sur","2000","Baja"),        ("F-003","Sur","2000","ACTIVO"),   # duplicado
    ("F-009","Sur","2000","Activo"),      ("F-016","Norte","","Activo"),     # monto vacio
]
df = pd.DataFrame(registro, columns=["folio","entidad","monto_apoyo","estatus"])
print(df.shape, "renglones x columnas")


(18, 4) renglones x columnas


## 2. La limpieza, paso a paso y documentada

Tres arreglos, cada uno una línea que se vuelve a correr:

1. **Texto a número.** Los montos ampliados vienen como «5,000» con coma; se quita la coma y se convierten a número. Lo que quede vacío se marca como faltante.
2. **Un folio, una vez.** Se quitan los folios repetidos, para no contar dos veces al mismo beneficiario.
3. **Fuera lo faltante.** El registro sin monto no entra al promedio; se reporta aparte como cobertura.


In [2]:
# 1. texto a numero (quita la coma de miles; lo no numerico queda como faltante)
df["monto_limpio"] = pd.to_numeric(df["monto_apoyo"].str.replace(",", "", regex=False), errors="coerce")

# 2. un folio, una vez  +  3. fuera lo faltante
validos = df.drop_duplicates(subset="folio", keep="first").dropna(subset=["monto_limpio"])

promedio_correcto = validos["monto_limpio"].mean()
print(f"Beneficiarios unicos con monto: {len(validos)}")
print(f"Promedio corregido: {promedio_correcto:,.0f}")
print(f"(cobertura: {len(validos)} de {df['folio'].nunique()} folios con monto valido)")


Beneficiarios unicos con monto: 15
Promedio corregido: 3,200
(cobertura: 15 de 16 folios con monto valido)


## 3. La prueba de reproducibilidad

La prueba de fuego es reiniciar y correr todo. Aquí la dejamos automática: la celda revisa que el resultado sea 3,200. Si alguien cambia un paso y rompe el flujo, esta celda avisa.


In [3]:
assert round(promedio_correcto) == 3200, "El flujo no da 3,200: algo se rompio."
print("Reproducible: corre de cero, de arriba a abajo, y da 3,200.")


Reproducible: corre de cero, de arriba a abajo, y da 3,200.


## 4. Chat contra código

Un modelo de chat predice el siguiente texto; no opera los números, los estima. Por eso cuenta mal una tabla y multiplica mal números grandes. El código no adivina: opera. Estas tres respuestas del código son verificables; las del chat son ilustrativas de su típico error.


In [4]:
norte   = int((df["entidad"] == "Norte").sum())   # cuenta exacta de renglones del Norte
producto = 4827 * 3916                             # operacion grande, exacta

print(f"Beneficiarios del Norte  ->  chat: ~5              codigo: {norte}")
print(f"4,827 x 3,916            ->  chat: ~18.9 millones  codigo: {producto:,}")
print(f"Promedio del apoyo       ->  chat: ~2,500          codigo: {round(promedio_correcto):,}")


Beneficiarios del Norte  ->  chat: ~5              codigo: 6
4,827 x 3,916            ->  chat: ~18.9 millones  codigo: 18,902,532
Promedio del apoyo       ->  chat: ~2,500          codigo: 3,200


## 5. El agente de código, y por qué dentro del notebook

Esta secuencia documentada y re-corrible es exactamente lo que produce un agente de código: le pides el análisis y escribe y ejecuta el código. La diferencia con pedirle el número a un chat es que aquí **puedes leer cada paso, cambiarlo y volver a correrlo**. El entorno reproducible es lo que convierte al agente de caja negra en algo auditable.

Esa es la idea de fondo del módulo, en una línea: no le pidas a la IA el número, pídele el código que lo calcula, y córrelo tú en un entorno donde se pueda revisar y repetir.

## Cierre

Ya tienen el flujo completo: explorar, depurar y verificar (Sesión 7), documentar y reproducir (hoy). La Sesión 9 lo aplica de principio a fin sobre un registro más grande, y la rúbrica técnica evalúa justo esto: un entregable que otra persona puede volver a correr y obtener el mismo resultado.
